# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LeylaAghayeva1/ml-search-engineering/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*
### Distribution analysis

Before defining the rule, I examined the distributions of the main signals.

The selected signals are:
- days_since_last_update
- search_volume
- impressions_90d

These variables are expected to have long-tailed distributions, where most pages have relatively small values and a smaller number of pages contain much larger values.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from pathlib import Path

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

cols = [
    "days_since_last_update",
    "search_volume",
    "impressions_90d"
]

for c in cols:
    print("="*50)
    print(c)
    print(df[c].describe())

    print("\nQuartiles")
    print(pd.qcut(
        df[c],
        4,
        duplicates="drop"
    ).value_counts())

    print("\n")

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*
### Signal Test 1

**Hypothesis:** Older pages should generally receive more refresh attention.

**Verdict: MIXED**

The bucket analysis does not show a consistent relationship between content age and impressions. Pages with a moderate time since the last update have the highest average impressions, while the oldest pages have much lower average impressions and search volume. This suggests that content age is an important signal but should not be used alone.

---

### Signal Test 2

**Hypothesis:** Higher search volume indicates more potential value from refreshing content.

**Verdict: MIXED**

Higher search volume does not consistently correspond to higher average impressions in this dataset. In addition, average CTR decreases as search volume increases. Search volume is still useful for prioritization, but it should be combined with other signals such as content freshness and visibility.

---

### Signal Test 3

**Hypothesis:** Pages with meaningful impressions represent existing visibility that could benefit from updates.

**Verdict: CONFIRMED**

Pages with higher impression counts already have search visibility, making them reasonable candidates for a content refresh. Existing visibility suggests there is an opportunity to improve performance through updated content.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
signals = [
    "days_since_last_update",
    "search_volume",
    "impressions_90d"
]

for s in signals:

    print("="*60)
    print(s)

    buckets = (
        df.assign(bucket=pd.qcut(df[s],5,duplicates="drop"))
          .groupby("bucket")
          .agg(
              n=("content_id","count"),
              avg_ctr=("ctr","mean"),
              avg_position=("avg_position","mean")
          )
    )

    print(buckets)

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### The flag-linked test

FlyRank's refresh logic assumes that content becomes a better refresh candidate as it gets older. To test this assumption, I examined `days_since_last_update` by dividing the data into buckets and comparing the average impressions and search volume.

**Verdict: MIXED**

The analysis shows that content age alone is not a strong predictor of refresh priority. Pages with a moderate time since the last update have the highest average impressions, while the oldest pages have lower impressions and search volume. This suggests that FlyRank's refresh signal should be combined with other signals such as search demand and existing visibility instead of relying on age alone.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
audit = (
    df.assign(
        bucket=pd.qcut(
            df["days_since_last_update"],
            5,
            duplicates="drop"
        )
    )
    .groupby("bucket")
    .agg(
        n=("content_id","count"),
        mean_impressions=("impressions_90d","mean"),
        mean_search_volume=("search_volume","mean")
    )
)

print(audit)

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The analysis suggests that no single signal is sufficient for identifying refresh candidates. Content age is useful but should be combined with search demand and existing visibility.

For the baseline, I use multiple transparent signals rather than relying on one metric alone. This produces a more balanced and explainable ranked queue while avoiding label leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Average search volume",
        "Average impressions",
        "Average days since update",
        "Average CTR"
    ],
    "Value": [
        len(df),
        round(df["search_volume"].mean(), 2),
        round(df["impressions_90d"].mean(), 2),
        round(df["days_since_last_update"].mean(), 2),
        round(df["ctr"].mean(), 3)
    ]
})

print(summary)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.